In [36]:
from lmut import LMUT
from dqn import DQNAgent, DQNNetwork, play_episode
from visualization import show_animation
import gymnasium as gym
import numpy as np
import os


In [18]:
models_dir = "models"
dqn_path = os.path.join(models_dir, "dqn_cartpole")
lmut_path = os.path.join(models_dir, "lmut")


In [3]:
env = gym.make("CartPole-v1", render_mode="rgb_array")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

print(state_dim, n_actions)


4 2


In [4]:
agent = DQNAgent(state_dim, n_actions)
agent.load(dqn_path)


In [5]:
episode_reward, episode_steps, _ = play_episode(env, agent)
print(f"Recompensa del episodio: {episode_reward}")
print(f"Pasos del episodio: {episode_steps}")


c:\Users\malos\Documents\GitHub\XRL\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Recompensa del episodio: 500.0
Pasos del episodio: 500


In [10]:
def collect_qs(env: gym.Env, agent: DQNAgent, episodes: int = 100, epsilon: float = 0.1):
	qs = []    

	for _ in range(episodes):
		state, _ = env.reset()
		done = False

		while not done:
			action = agent.epsilon_greedy(state, epsilon)
			q_values = agent.predict(state)

			qs.extend(q_values)

			state, _, terminated, truncated, _ = env.step(action)
			done = terminated or truncated

	return np.array(qs)


In [11]:
qs = collect_qs(env, agent)
print(f"min={qs.min():.2f}  max={qs.max():.4f}  mean={qs.mean():.4f}  std={qs.std():.4f}")
print(f"percentiles 1/50/99: {np.percentile(qs, [1, 50, 99])}")


min=-52.83  max=201.5200  mean=183.0562  std=8.1728
percentiles 1/50/99: [169.7753685  185.05003357 192.65179352]


In [15]:
def active_play_mimic(env, agent: DQNAgent, mimic_lmut: LMUT, episodes: int = 200, train_every: int = 1000, epsilon_start: float = 1.0, epsilon_end: float = 0.1, decay: float = 0.995):
	epsilon = epsilon_start
	buffer = []

	for ep in range(episodes):
		state, _ = env.reset()
		done = False
		while not done:
			action = agent.epsilon_greedy(state, epsilon)
			teacher_q = agent.predict(state)[action]

			buffer.append((state, action, teacher_q))

			next_state, _, terminated, truncated, _ = env.step(action)
			done = terminated or truncated
			state = next_state

			if len(buffer) >= train_every:
				for s, a, tq in buffer:
					mimic_lmut.add_transition(s, a, tq)
				stats = mimic_lmut.update_all_leaves()
				mse = mimic_lmut.compute_current_mse()

				print(
					f"Episode {ep+1} | "
					f"Leaves: {stats['leaf_count']} | "
					# f"AvgLoss: {stats['avg_loss']:.4f} | "
					f"MSE: {mse:.4f} | "
					f"Splits: {stats['splits']} | "
					f"ε: {epsilon:.3f}"
				)
				
				buffer.clear()

		epsilon = max(epsilon_end, epsilon * decay)
		

In [12]:
mimic = LMUT(state_dim, n_actions, q_mean=qs.mean(), q_std=qs.std())


In [17]:
active_play_mimic(env, agent, mimic, episodes=100)


Episode 42 | Leaves: 4 | MSE: 6964.7021 | Splits: 4 | ε: 0.814
Episode 74 | Leaves: 8 | MSE: 6241.4751 | Splits: 5 | ε: 0.694
Episode 97 | Leaves: 13 | MSE: 5194.7061 | Splits: 4 | ε: 0.618


In [20]:
mimic.print_tree(0)


=== Tree for action 0 ===
[Node] depth=0 | split: feature 2 < 0.0713
  left:
    [Node] depth=1 | split: feature 3 < -0.5486
      left:
        [Node] depth=2 | split: feature 2 < -0.1303
          left:
            [Leaf] depth=3 | y = [ 0.12456903 -0.5104677   0.42838165  0.15596776] * s + -0.3616
          right:
            [Node] depth=3 | split: feature 3 < -1.0816
              left:
                [Leaf] depth=4 | y = [ 0.12943754 -0.48646575  0.42394206  0.11808065] * s + -0.3368
              right:
                [Leaf] depth=4 | y = [ 0.12959652 -0.4859022   0.42388865  0.11750817] * s + -0.3362
      right:
        [Leaf] depth=2 | y = [ 0.13234316 -0.45872512  0.4200674   0.0763022 ] * s + -0.3169
  right:
    [Node] depth=1 | split: feature 3 < 0.6435
      left:
        [Node] depth=2 | split: feature 2 < 0.1348
          left:
            [Leaf] depth=3 | y = [ 0.13675325 -0.4258172   0.41062668  0.01127215] * s + -0.3567
          right:
            [Leaf] depth=3

In [19]:
mimic.save(lmut_path)


In [23]:
lmut2 = LMUT.load(lmut_path)


In [24]:
lmut2.print_tree(0)



=== Tree for action 0 ===
[Node] depth=0 | split: feature 2 < 0.0713
  left:
    [Node] depth=1 | split: feature 3 < -0.5486
      left:
        [Node] depth=2 | split: feature 2 < -0.1303
          left:
            [Leaf] depth=3 | y = [ 0.12456903 -0.5104677   0.42838165  0.15596776] * s + -0.3616
          right:
            [Node] depth=3 | split: feature 3 < -1.0816
              left:
                [Leaf] depth=4 | y = [ 0.12943754 -0.48646575  0.42394206  0.11808065] * s + -0.3368
              right:
                [Leaf] depth=4 | y = [ 0.12959652 -0.4859022   0.42388865  0.11750817] * s + -0.3362
      right:
        [Leaf] depth=2 | y = [ 0.13234316 -0.45872512  0.4200674   0.0763022 ] * s + -0.3169
  right:
    [Node] depth=1 | split: feature 3 < 0.6435
      left:
        [Node] depth=2 | split: feature 2 < 0.1348
          left:
            [Leaf] depth=3 | y = [ 0.13675325 -0.4258172   0.41062668  0.01127215] * s + -0.3567
          right:
            [Leaf] depth=3

In [26]:
def evaluate_fidelity(env: gym.Env, teacher: DQNAgent, mimic_lmut: LMUT, num_samples: int = 1000):
	mae_sum = 0.0
	mse_sum = 0.0
	count = 0
	correct = 0

	state, _ = env.reset()
	for _ in range(num_samples):
		q_all = teacher.predict(state)
		teacher_action = int(np.argmax(q_all))
		teacher_q = q_all[teacher_action]
	
		mimic_qs = np.array(mimic_lmut.predict_all_actions(state))
		
		mimic_action = np.argmax(mimic_qs)
		mimic_q = mimic_qs[teacher_action]

		if teacher_action == mimic_action:
			correct += 1

		error = teacher_q - mimic_q
		mae_sum += abs(error)
		mse_sum += error ** 2
		count += 1

		next_state, _, terminated, truncated, _ = env.step(teacher_action)
		if terminated or truncated:
			state, _ = env.reset()
		else:
			state = next_state

	mae = mae_sum / count
	rmse = (mse_sum / count) ** 0.5
	accuracy = correct / count
	return mae, rmse, accuracy


In [28]:
mae, rmse, acc = evaluate_fidelity(env, agent, mimic, num_samples=10000)
print(f"Fidelity: MAE = {mae:.4f}, RMSE = {rmse:.4f}, Action Accuracy = {acc:.2%}")


Fidelity: MAE = 5.8217, RMSE = 6.6218, Action Accuracy = 49.91%


In [33]:
def play_episode_lmut(env: gym.Env, lmut: LMUT):
    frames = []
    state, _ = env.reset()

    episode_steps = episode_reward = 0
    done = False
    while not done:
        frames.append(env.render())
        q_values = np.array(lmut.predict_all_actions(state))
        action = int(np.argmax(q_values))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        episode_steps += 1
        episode_reward += reward

        state = next_state
    return episode_reward, episode_steps, frames



In [37]:
episode_reward, episode_steps, frames = play_episode_lmut(env, lmut2)
print(f"Recompensa del episodio: {episode_reward}")
print(f"Pasos del episodio: {episode_steps}")


Recompensa del episodio: 10.0
Pasos del episodio: 10


In [39]:
show_animation(frames)
